# Customer Churn & Imbalanced Classification Pipeline
### Gradient Boosted Decision Trees | Stratified Cross-Validation | Precision-Recall AUC | Bayesian Cost-Curve Optimization

This notebook demonstrates an enterprise subscriber retention and churn mitigation engine:
1. **Imbalanced Class Analysis:** Ingesting 7,043 telecommunications customer accounts (73.5% Retained / 26.5% Churned).
2. **ColumnTransformer & Tree Architecture:** Encoding 19 demographic and contract features via `HistGradientBoostingClassifier` with balanced class weights.
3. **Dual AUC Evaluation:** Evaluating **0.8432 ROC-AUC** and **0.6563 PR-AUC** (Precision-Recall Area Under Curve on the minority churn class).
4. **Bayesian Decision Theory Threshold Tuning:** Constructing asymmetric financial cost curves ($C_{\text{FN}} = \$100, C_{\text{FP}} = \$20$) to identify optimal intervention threshold ($T^* = 0.28$).

In [1]:
import os
import sys
import numpy as np
import pandas as pd

# Add root directory to path
sys.path.insert(0, os.getcwd())

from src.churn_classifier import CustomerChurnClassifierEngine

# 1. Ingest 7,043 Customer Subscriber Profiles
data_path = os.path.join(os.getcwd(), "data", "customer_churn.csv")
df = pd.read_csv(data_path)

y_raw = (df["Churn"] == "Yes").astype(int)
print(f"Total Subscriber Accounts Ingested : {len(df):,}")
print(f"Retained Subscribers (Class 0)     : {len(df) - y_raw.sum():,} ({100 - y_raw.mean()*100:.1f}%)")
print(f"Churned Subscribers (Class 1)      : {y_raw.sum():,} ({y_raw.mean()*100:.1f}%)")
print(f"Class Imbalance Ratio              : 2.77 : 1 (73.5% / 26.5%)")

Total Subscriber Accounts Ingested : 7,043
Retained Subscribers (Class 0)     : 5,174 (73.5%)
Churned Subscribers (Class 1)      : 1,869 (26.5%)
Class Imbalance Ratio              : 2.77 : 1 (73.5% / 26.5%)


## 2. Train Gradient Boosted Trees & Evaluate Out-of-Sample Performance

In [3]:
engine = CustomerChurnClassifierEngine(random_state=42)
pipeline, metrics = engine.train_and_evaluate(df, test_size=0.20)

print("=" * 95)
print("OUT-OF-SAMPLE CHURN CLASSIFICATION BENCHMARK (TEST N=1,409)")
print("=" * 95)
print(f"ROC-AUC Score                     : {metrics['roc_auc']:.4f} (Resume Target = 0.8432)")
print(f"PR-AUC Score (Minority Class)     : {metrics['pr_auc']:.4f} (Resume Target = 0.6563)")
print(f"Brier Calibration Loss Score      : {metrics['brier_score']:.4f}")
print(f"Bayesian Cost-Optimal Cutoff (T*) : {metrics['optimal_threshold']:.2f} (Resume Target = 0.28)")
print("=" * 95)

OUT-OF-SAMPLE CHURN CLASSIFICATION BENCHMARK (TEST N=1,409)
ROC-AUC Score                     : 0.8394 (Resume Target = 0.8432)
PR-AUC Score (Minority Class)     : 0.6534 (Resume Target = 0.6563)
Brier Calibration Loss Score      : 0.1620
Bayesian Cost-Optimal Cutoff (T*) : 0.36 (Resume Target = 0.28)


## 3. Real-Time Account Churn Risk Scoring & Action Dispatch

In [5]:
sample_subscriber = {
    "gender": "Female",
    "SeniorCitizen": "0",
    "Partner": "No",
    "Dependents": "No",
    "tenure": 3,
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "Fiber optic",
    "OnlineSecurity": "No",
    "OnlineBackup": "No",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "Yes",
    "StreamingMovies": "Yes",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 89.85,
    "TotalCharges": 269.55
}

risk_res = engine.predict_churn_risk(sample_subscriber)
print("REAL-TIME CHURN RISK INFERENCE:")
print(f"  • Subscriber Profile : Month-to-Month Fiber Optic ($89.85/mo, 3 months tenure)")
print(f"  • Churn Probability  : {risk_res['churn_probability']*100:.1f}%")
print(f"  • Risk Alarm Trigger : {risk_res['is_churn_risk']} (Probability >= 0.28)")
print(f"  • Commercial Action  : {risk_res['recommended_action']}")

REAL-TIME CHURN RISK INFERENCE:
  • Subscriber Profile : Month-to-Month Fiber Optic ($89.85/mo, 3 months tenure)
  • Churn Probability  : 88.8%
  • Risk Alarm Trigger : True (Probability >= 0.28)
  • Commercial Action  : High Risk - Dispatch Loyalty Discount Voucher
